# GolStats - Scale Pipeline

This notebook expands the GolStats data pipeline from the initial development
dataset to the complete FIFA World Cup 2022 competition.

The previous notebooks were developed and validated using a controlled subset
of 10 matches. This allowed the Bronze, Silver, and Gold transformations to be
tested and validated before increasing the data volume.

The objective of this notebook is to scale the ingestion process to all
available matches in the competition while preserving the existing Medallion
Architecture.

The pipeline will continue to follow the same structure:

![image_1788534119915.png](./image_1788534119915.png "image_1788534119915.png")

The scaling process will validate that the existing transformations remain
consistent when applied to the complete competition dataset.

The final dataset will contain all 64 FIFA World Cup 2022 matches.


## 1. Define scaling scope

The initial GolStats implementation used 10 matches as a controlled development
dataset.

For the scaling phase, the ingestion scope will be expanded to all matches
available for the FIFA World Cup 2022 competition.

The competition and season identifiers remain unchanged:

* Competition: FIFA World Cup
* Season: 2022
* Competition ID: 43
* Season ID: 106

Expected scope:

* 64 matches
* Complete competition match metadata
* Event data for every available match

The existing Bronze, Silver, and Gold transformation logic will be reused
without changing the analytical definitions already validated.


##2. Retrieve complete match metadata

In [0]:
import requests

BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

COMPETITION_ID = 43
SEASON_ID = 106

matches_url = f"{BASE_URL}/matches/{COMPETITION_ID}/{SEASON_ID}.json"

response = requests.get(matches_url)
response.raise_for_status()

matches_full = response.json()

print(f"Total matches available: {len(matches_full)}")

## 3. Validate competition scope

Before downloading the event data, the complete match metadata is validated
to confirm that the expected number of matches is available.

This prevents the ingestion process from proceeding with an incomplete or
unexpected competition dataset.

The expected number of matches for the FIFA World Cup 2022 is 64.


In [0]:
assert len(matches_full) == 64, (
    f"Expected 64 matches, but found {len(matches_full)}"
)

print("Competition scope validated successfully.")

## 4. Build complete match metadata

The complete competition metadata is transformed into a simplified dataset
containing the attributes required by downstream analytical processes.

The dataset will include:

* `match_id`
* `match_date`
* `home_team`
* `away_team`
* `home_score`
* `away_score`

This metadata will serve as the reference dataset for the complete competition
and will also be used later to validate the scaled event ingestion.


In [0]:
matches_clean = []

for match in matches_full:
    matches_clean.append({
        "match_id": match.get("match_id"),
        "match_date": match.get("match_date"),
        "home_team": match.get("home_team", {}).get("home_team_name"),
        "away_team": match.get("away_team", {}).get("away_team_name"),
        "home_score": match.get("home_score"),
        "away_score": match.get("away_score")
    })

matches_complete_df = spark.createDataFrame(matches_clean)

display(
    matches_complete_df
    .orderBy("match_date", "match_id")
)

In [0]:
from pyspark.sql.functions import count, countDistinct

display(
    matches_complete_df.agg(
        count("*").alias("total_matches"),
        countDistinct("match_id").alias("unique_match_ids")
    )
)

## 5. Configure raw data storage

The raw StatsBomb event files are stored in a Unity Catalog Volume.

The same storage location used during the initial ingestion will be reused
for the complete competition dataset.

Keeping the same raw storage location allows the scaling process to preserve
the existing 10 development files and incrementally add the remaining matches.

The raw event files are stored under:

`/Volumes/golstats/bronze/raw_files`



In [0]:
VOLUME_PATH = "/Volumes/golstats/bronze/raw_files"

print(f"Raw data path: {VOLUME_PATH}")

## 6. Identify already ingested matches

The initial ingestion process downloaded event data for 10 matches as a
development dataset.

Before scaling the pipeline, the existing raw files will be compared against
the complete competition metadata.

Only missing matches will be downloaded.

This approach avoids unnecessary network requests, prevents duplicated files,
and makes the scaling process incremental and reproducible.

The expected result is:

* 64 matches available in the competition metadata.
* 10 matches already ingested.
* 54 matches requiring ingestion.


In [0]:
import glob
import re

existing_files = glob.glob(
    f"{VOLUME_PATH}/eventos_*.json"
)

existing_match_ids = sorted([
    int(re.search(r"eventos_(\d+)\.json", file).group(1))
    for file in existing_files
])

print(f"Existing event files: {len(existing_match_ids)}")
print(f"Existing match IDs: {existing_match_ids}")

In [0]:
all_match_ids = sorted([
    match["match_id"]
    for match in matches_full
])

missing_match_ids = sorted(
    set(all_match_ids) - set(existing_match_ids)
)

print(f"Total competition matches: {len(all_match_ids)}")
print(f"Already ingested: {len(existing_match_ids)}")
print(f"Missing matches: {len(missing_match_ids)}")
print(f"Missing match IDs: {missing_match_ids}")

## 7. Download missing event data

The missing event files will now be downloaded from the StatsBomb Open Data
repository.

Only the 54 matches identified as missing will be processed.

The existing 10 event files will not be modified or downloaded again.

Each event file will be stored using the same naming convention established
during the initial ingestion:

`eventos_<match_id>.json`

This ensures that the complete raw dataset remains consistent and can be
processed by the existing Bronze ingestion logic.


In [0]:
import requests

for match_id in missing_match_ids:

    events_url = f"{BASE_URL}/events/{match_id}.json"

    response = requests.get(events_url)
    response.raise_for_status()

    file_path = f"{VOLUME_PATH}/eventos_{match_id}.json"

    with open(file_path, "w") as f:
        f.write(response.text)

    print(f"Match {match_id} ingested successfully")

## 8. Validate raw event files

After the incremental ingestion, the raw storage location is validated to
ensure that an event file exists for every match in the complete competition.

This validation compares:

* The expected match IDs from the official competition metadata.
* The match IDs extracted from the raw event filenames.

The expected result is:

* 64 expected matches.
* 64 event files.
* 0 missing event files.
* 0 unexpected event files.

This confirms that the Raw layer is complete before rebuilding the Bronze layer.


In [0]:
import glob
import re

event_files = glob.glob(
    f"{VOLUME_PATH}/eventos_*.json"
)

raw_match_ids = sorted([
    int(re.search(r"eventos_(\d+)\.json", file).group(1))
    for file in event_files
])

expected_match_ids = sorted([
    match["match_id"]
    for match in matches_full
])

missing_raw_files = sorted(
    set(expected_match_ids) - set(raw_match_ids)
)

unexpected_raw_files = sorted(
    set(raw_match_ids) - set(expected_match_ids)
)

print(f"Expected matches: {len(expected_match_ids)}")
print(f"Raw event files: {len(raw_match_ids)}")
print(f"Missing event files: {len(missing_raw_files)}")
print(f"Unexpected event files: {len(unexpected_raw_files)}")

## 9. Rebuild the Bronze layer

The Raw layer now contains event files for all 64 FIFA World Cup 2022 matches.

The Bronze layer will therefore be rebuilt using the complete set of raw JSON
files.

The existing Bronze transformation logic will be reused without changing its
business meaning.

The process will:

1. Read all StatsBomb event JSON files from the Raw storage location.
2. Preserve the original event structure.
3. Add the source file path for data lineage.
4. Extract the `match_id` from the source filename.
5. Persist the complete dataset as a Delta table.

The resulting table will contain event data for all 64 matches:

`golstats.bronze.eventos_statsbomb`

The expected grain remains:

**One row = one StatsBomb event.**


In [0]:
df_events = (
    spark.read
    .option("multiLine", True)
    .json(f"{VOLUME_PATH}/eventos_*.json")
)

display(df_events.head())

In [0]:
from pyspark.sql.functions import col, regexp_extract

df_events_with_lineage = (
    df_events
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )
    .withColumn(
        "match_id",
        regexp_extract(
            col("source_file"),
            r"eventos_(\d+)\.json",
            1
        ).cast("long")
    )
)

display(df_events_with_lineage.head())

In [0]:
from pyspark.sql.functions import count, countDistinct

display(
    df_events_with_lineage.agg(
        count("*").alias("total_events"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("id").alias("unique_event_ids")
    )
)

## 10. Persist the complete Bronze layer

The complete event dataset has passed the initial structural validation.

The dataset contains event data for all 64 matches, with unique event
identifiers across the complete competition.

The validated DataFrame will now replace the previous development Bronze
table.

The table will continue to use Delta format and the same Unity Catalog
location:

`golstats.bronze.eventos_statsbomb`

The overwrite is intentional because the previous table represented only the
10-match development dataset.

After persistence, the table will be reloaded from Unity Catalog and validated
again to ensure that the stored Delta dataset matches the validated DataFrame.


In [0]:
BRONZE_TABLE = "golstats.bronze.eventos_statsbomb"

(
    df_events_with_lineage
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print("Bronze table persisted successfully.")

### 10.1 Recreate the Bronze Delta table

The existing Bronze table contains metadata associated with the previous
10-match development dataset.

Because the scaled dataset replaces that development version, the existing
Delta table will be removed before persisting the complete dataset.

The raw JSON files remain unchanged.

Only the Bronze table definition is recreated.

This ensures that the complete 64-match dataset can be persisted with its
current schema and metadata.


In [0]:
%sql
DROP TABLE IF EXISTS golstats.bronze.eventos_statsbomb;

In [0]:
BRONZE_TABLE = "golstats.bronze.eventos_statsbomb"

(
    df_events_with_lineage
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print("Bronze table persisted successfully.")

In [0]:
from pyspark.sql.functions import count, countDistinct

df_bronze = spark.table("golstats.bronze.eventos_statsbomb")

display(
    df_bronze.agg(
        count("*").alias("total_events"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("id").alias("unique_event_ids")
    )
)

## 11. Rebuild the Silver layer

The Bronze layer has now been successfully scaled to all 64 FIFA World Cup
2022 matches.

The existing Silver transformation will be applied to the complete Bronze
dataset.

The transformation logic remains unchanged because it was previously
developed and validated using the 10-match development dataset.

The Silver layer will:

* Flatten the required nested StatsBomb fields.
* Standardize column names.
* Extract event attributes.
* Normalize pass outcomes.
* Preserve event-level granularity.
* Include the complete 64-match dataset.

The resulting table will replace the previous 10-match Silver dataset:

`golstats.silver.eventos`

The expected grain remains:

**One row = one StatsBomb event.**


##1.1 Read the complete Bronze dataset


In [0]:
df_bronze = spark.table("golstats.bronze.eventos_statsbomb")

display(df_bronze)

In [0]:
from pyspark.sql.functions import count, countDistinct

display(
    df_bronze.agg(
        count("*").alias("total_events"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("id").alias("unique_event_ids")
    )
)

### 11.2 Transform Bronze into Silver

The Silver transformation previously developed for the 10-match development
dataset will now be applied to the complete Bronze dataset.

The transformation preserves the same analytical structure and business logic
validated during the development phase.

The main transformations include:

* Standardizing event identifiers and timestamps.
* Extracting team and player information.
* Flattening event classification fields.
* Extracting event coordinates.
* Extracting pass attributes and destinations.
* Extracting shot attributes and destinations.
* Normalizing pass outcomes.
* Preserving the event-level grain.

No new business rules are introduced during the scaling process.


In [0]:
from pyspark.sql.functions import col

df_silver = (
    df_bronze
    .select(
        # Event identification
        col("id").alias("event_id"),
        col("match_id"),
        col("index"),
        col("timestamp"),

        # Time information
        col("period"),
        col("minute"),
        col("second"),

        # Event classification
        col("type.name").alias("event_type"),

        # Team information
        col("team.id").alias("team_id"),
        col("team.name").alias("team"),

        # Player information
        col("player.id").alias("player_id"),
        col("player.name").alias("player"),
        col("position.name").alias("position"),

        # Possession information
        col("possession"),
        col("possession_team.id").alias("possession_team_id"),
        col("possession_team.name").alias("possession_team"),

        # Event location
        col("location").getItem(0).alias("x"),
        col("location").getItem(1).alias("y"),

        # General event context
        col("play_pattern.name").alias("play_pattern"),
        col("under_pressure"),
        col("counterpress"),

        # Pass attributes
        col("pass.length").alias("pass_length"),
        col("pass.angle").alias("pass_angle"),
        col("pass.outcome.name").alias("pass_outcome"),
        col("pass.recipient.id").alias("pass_recipient_id"),
        col("pass.recipient.name").alias("pass_recipient"),
        col("pass.technique.name").alias("pass_technique"),
        col("pass.height.name").alias("pass_height"),
        col("pass.cross").alias("pass_cross"),
        col("pass.cut_back").alias("pass_cut_back"),
        col("pass.through_ball").alias("pass_through_ball"),
        col("pass.switch").alias("pass_switch"),
        col("pass.goal_assist").alias("pass_goal_assist"),
        col("pass.end_location").getItem(0).alias("pass_end_x"),
        col("pass.end_location").getItem(1).alias("pass_end_y"),

        # Shot attributes
        col("shot.statsbomb_xg").alias("shot_xg"),
        col("shot.outcome.name").alias("shot_outcome"),
        col("shot.technique.name").alias("shot_technique"),
        col("shot.body_part.name").alias("shot_body_part"),
        col("shot.first_time").alias("shot_first_time"),
        col("shot.one_on_one").alias("shot_one_on_one"),
        col("shot.end_location").getItem(0).alias("shot_end_x"),
        col("shot.end_location").getItem(1).alias("shot_end_y")
    )
)

In [0]:
from pyspark.sql.functions import when

df_silver = (
    df_silver
    .withColumn(
        "pass_outcome",
        when(
            (col("event_type") == "Pass") &
            col("pass_outcome").isNull(),
            "Complete"
        ).otherwise(col("pass_outcome"))
    )
)

## 12. Validate scaled Silver dataset

The complete Silver dataset is now validated before persistence.

Because the transformation logic was already validated during development,
the scaling validation focuses on structural integrity.

The validation checks:

* Total number of events.
* Number of matches.
* Number of unique event identifiers.

The expected result is:

* 234,637 events.
* 64 matches.
* 234,637 unique event identifiers.

This confirms that the Bronze-to-Silver transformation preserved the complete
event population during scaling.


In [0]:
from pyspark.sql.functions import count, countDistinct

display(
    df_silver.agg(
        count("*").alias("total_events"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("event_id").alias("unique_event_ids")
    )
)

## 13. Persist the complete Silver layer

The complete Silver dataset has passed the structural validation.

The validated DataFrame will now replace the previous 10-match Silver table.

The existing Silver table is intentionally recreated because the previous
version represented only the development dataset.

The resulting Delta table will contain the complete event-level dataset for
all 64 FIFA World Cup 2022 matches:

`golstats.silver.eventos`

The analytical grain remains:

**One row = one StatsBomb event.**


In [0]:
%sql
DROP TABLE IF EXISTS golstats.silver.eventos;

In [0]:
SILVER_TABLE = "golstats.silver.eventos"

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table persisted successfully.")

##13.1 Validate persisted Silver

In [0]:
from pyspark.sql.functions import count, countDistinct

df_silver_persisted = spark.table("golstats.silver.eventos")

display(
    df_silver_persisted.agg(
        count("*").alias("total_events"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("event_id").alias("unique_event_ids")
    )
)

## 14. Rebuild the Gold layer

The Bronze and Silver layers have now been successfully scaled to the complete
64-match competition dataset.

The Gold layer will now be rebuilt using the complete Silver dataset.

The analytical definitions developed and validated during the development phase
will remain unchanged.

The Gold layer contains three analytical datasets:

* `golstats.gold.partidos` — one row per match.
* `golstats.gold.estadisticas_equipo` — one row per team per match.
* `golstats.gold.estadisticas_jugador` — one row per player per match.

The objective of this phase is to verify that the existing analytical logic
scales correctly from the 10-match development dataset to the complete
competition.


In [0]:
df_silver = spark.table("golstats.silver.eventos")

display(
    df_silver.agg(
        count("*").alias("total_events"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("event_id").alias("unique_event_ids")
    )
)

## 14.1 Build team-match statistics

The first Gold dataset will aggregate event-level data into team-level match
statistics.

The analytical grain remains:

**One row = one team in one match.**

The following metrics will be calculated:

* `total_events`
* `passes`
* `completed_passes`
* `pass_completion_pct`
* `shots`
* `goals`
* `xg`
* `pressures`
* `ball_recoveries`
* `dispossessions`

The aggregation logic is the same as the one previously validated with the
10-match development dataset.

The difference is that the transformation will now operate on the complete
64-match Silver dataset.


In [0]:
from pyspark.sql.functions import col, count, sum, when, round

df_team_match = (
    df_silver
    .groupBy(
        "match_id",
        "team"
    )
    .agg(
        count("*").alias("total_events"),

        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                True
            )
        ).alias("goals"),

        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        count(
            when(col("event_type") == "Pressure", True)
        ).alias("pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

display(df_team_match)

## 14.2 Validate team-match grain

The team-level Gold dataset must contain exactly one record for each team and
match combination.

Since every FIFA World Cup match contains two teams, the complete 64-match
dataset should produce:

* 64 unique matches.
* 128 team-match records.
* 128 unique team-match combinations.
* Exactly 2 teams per match.

This validation confirms that the aggregation preserves the intended Gold
grain before the dataset is persisted.


In [0]:
from pyspark.sql.functions import count, countDistinct

display(
    df_team_match.agg(
        count("*").alias("total_team_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("team").alias("total_teams")
    )
)

In [0]:
display(
    df_team_match
    .groupBy("match_id")
    .agg(
        countDistinct("team").alias("teams_per_match")
    )
    .filter(col("teams_per_match") != 2)
)

## 14.3 Validate team-match metrics

The team-match aggregation has passed the structural grain validation.

The next step is to verify that the aggregated metrics are mathematically
consistent with the underlying Silver event dataset.

The following metrics will be compared between Silver and the aggregated
team-match dataset:

* `passes`
* `shots`
* `goals`
* `xg`
* `pressures`
* `ball_recoveries`
* `dispossessions`

The totals calculated at Gold level must match the corresponding totals
calculated directly from Silver.

This validation ensures that no events were lost, duplicated, or incorrectly
classified during the aggregation process.

The pass completion percentage will be validated separately because it is a
derived metric calculated from `passes` and `completed_passes`.


In [0]:
from pyspark.sql.functions import sum, col

silver_totals = (
    df_silver
    .agg(
        sum(
            when(col("event_type") == "Pass", 1).otherwise(0)
        ).alias("passes"),

        sum(
            when(col("event_type") == "Shot", 1).otherwise(0)
        ).alias("shots"),

        sum(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                1
            ).otherwise(0)
        ).alias("goals"),

        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        sum(
            when(col("event_type") == "Pressure", 1).otherwise(0)
        ).alias("pressures"),

        sum(
            when(col("event_type") == "Ball Recovery", 1).otherwise(0)
        ).alias("ball_recoveries"),

        sum(
            when(col("event_type") == "Dispossessed", 1).otherwise(0)
        ).alias("dispossessions")
    )
)

gold_totals = (
    df_team_match
    .agg(
        sum("passes").alias("passes"),
        sum("shots").alias("shots"),
        sum("goals").alias("goals"),
        sum("xg").alias("xg"),
        sum("pressures").alias("pressures"),
        sum("ball_recoveries").alias("ball_recoveries"),
        sum("dispossessions").alias("dispossessions")
    )
)

display(silver_totals)
display(gold_totals)

## 14.4 Validate pass completion percentage

The core team-match metrics have been validated against the Silver event dataset.

The remaining derived metric is `pass_completion_pct`.

This metric is calculated as:

`completed_passes / passes * 100`

The validation will verify that:

* The percentage is correctly calculated.
* No percentage is greater than 100%.
* No percentage is negative.
* Teams with zero passes do not produce invalid values.

This confirms that the derived KPI is mathematically consistent before the
Gold dataset is persisted.


In [0]:
from pyspark.sql.functions import min, max, count, when

display(
    df_team_match.agg(
        min("pass_completion_pct").alias("min_pass_completion_pct"),
        max("pass_completion_pct").alias("max_pass_completion_pct"),
        count(
            when(
                (col("pass_completion_pct") < 0) |
                (col("pass_completion_pct") > 100),
                True
            )
        ).alias("invalid_percentages"),
        count(
            when(
                (col("passes") == 0) &
                (col("pass_completion_pct") != 0),
                True
            )
        ).alias("zero_passes_with_nonzero_pct")
    )
)

## 14.5 Persist team-match Gold dataset

The team-match Gold dataset has passed all structural and metric validations.

It will now be persisted as a Delta table in the Gold layer.

The previous Gold table contains only the 10-match development dataset, so it
will be replaced by the complete 64-match dataset.

Target table:

`golstats.gold.estadisticas_equipo`

The resulting table will contain one row per team per match.


In [0]:
%sql
DROP TABLE IF EXISTS golstats.gold.estadisticas_equipo;

In [0]:
TEAM_GOLD_TABLE = "golstats.gold.estadisticas_equipo"

(
    df_team_match
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TEAM_GOLD_TABLE)
)

print(f"Gold team table persisted: {TEAM_GOLD_TABLE}")

In [0]:
df_team_gold = spark.table("golstats.gold.estadisticas_equipo")

display(
    df_team_gold.agg(
        count("*").alias("total_team_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("team").alias("total_teams"),
        countDistinct("match_id", "team").alias("unique_team_matches")
    )
)

## 14.6 Build player-match statistics

The second Gold dataset will aggregate the Silver event data into player-level
match statistics.

The analytical grain is:

**One row = one player in one match.**

The aggregation will be grouped by:

* `match_id`
* `team`
* `player_id`
* `player`

The `position` field will not be part of the grouping key because a player can
change position during a match. Including position would therefore create
multiple records for the same player-match combination.

The following metrics will be calculated:

* `total_events`
* `passes`
* `completed_passes`
* `pass_completion_pct`
* `shots`
* `goals`
* `xg`
* `pressures`
* `ball_recoveries`
* `dispossessions`

The aggregation logic is the same logic previously validated using the
10-match development dataset.

It will now be applied to the complete 64-match Silver dataset.


In [0]:
from pyspark.sql.functions import col, count, sum, when, round

df_player_match = (
    df_silver
    .groupBy(
        "match_id",
        "team",
        "player_id",
        "player"
    )
    .agg(
        count("*").alias("total_events"),

        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                True
            )
        ).alias("goals"),

        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        count(
            when(col("event_type") == "Pressure", True)
        ).alias("pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

display(df_player_match)

### 14.6.1 Investigate events without player attribution

During the initial inspection of the player-match aggregation, some records were
found with a valid team but without a player identifier.

These records should not be interpreted as player statistics.

Before filtering them from the Gold player dataset, the underlying event types
will be investigated to determine which StatsBomb events generate records
without player attribution.

This analysis will help distinguish expected event-level behavior from a data
quality issue.

The objective is to understand the source of these records before applying the
final player-level filtering rule.


In [0]:
display(
    df_silver
    .filter(col("player_id").isNull())
    .groupBy("event_type")
    .count()
    .orderBy(col("count").desc())
)

### 14.6.2 Filter events without player attribution

The investigation confirmed that events without player attribution correspond to
match-level or team-level events such as period starts, period ends, tactical
shifts, referee ball drops, starting lineups, and own-goal-for events.

These events are valid StatsBomb events, but they do not represent individual
player actions.

Therefore, the player Gold dataset will only include events with a valid
`player_id`.

This prevents match-level events from being incorrectly attributed to a
fictional player record.

The Silver dataset itself will remain unchanged because these events are valid
and should be preserved at the event level.


In [0]:
df_player_match = (
    df_silver
    .filter(col("player_id").isNotNull())
    .groupBy(
        "match_id",
        "team",
        "player_id",
        "player"
    )
    .agg(
        count("*").alias("total_events"),

        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                True
            )
        ).alias("goals"),

        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        count(
            when(col("event_type") == "Pressure", True)
        ).alias("pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

display(df_player_match)

### 14.6.3 Validate player-match grain

The player-match dataset has been filtered to include only events with a valid
player attribution.

The next validation confirms that the intended analytical grain is preserved:

**One row = one player in one match.**

The validation will check:

* Number of player-match records.
* Number of matches represented.
* Number of unique players.
* Number of duplicated player-match combinations.
* Remaining null player identifiers.

A player may appear in multiple matches, but the combination of
`match_id` and `player_id` must be unique.


In [0]:
from pyspark.sql.functions import count, countDistinct, col

display(
    df_player_match.agg(
        count("*").alias("total_player_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("player_id").alias("unique_players"),
        count(
            when(col("player_id").isNull(), True)
        ).alias("null_player_ids")
    )
)

In [0]:
display(
    df_player_match
    .groupBy("match_id", "player_id")
    .count()
    .filter(col("count") > 1)
)

### 14.6.4 Persist player-match Gold dataset

The player-match Gold dataset has passed the grain validation.

The complete dataset contains player statistics for all 64 FIFA World Cup 2022 matches.

The previous Gold table contains only the 10-match development dataset and will therefore be replaced.

The resulting Delta table will contain one row per player per match:

`golstats.gold.estadisticas_jugador`


In [0]:
%sql
DROP TABLE IF EXISTS golstats.gold.estadisticas_jugador;

In [0]:
PLAYER_GOLD_TABLE = "golstats.gold.estadisticas_jugador"

(
    df_player_match
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PLAYER_GOLD_TABLE)
)

print(f"Gold player table persisted: {PLAYER_GOLD_TABLE}")

## 15. Validate the Complete Gold Layer

The Gold layer has now been successfully rebuilt using the complete 64-match competition dataset.

Before proceeding to downstream analytics and visualization, the complete Gold layer must be validated to ensure that:

* all expected matches are present;
* team-level records have the correct granularity;
* player-level records have the correct granularity;
* no duplicate analytical records exist;
* key football metrics are internally consistent.

The validation process covers the three Gold tables:

* golstats.gold.partidos
* golstats.gold.estadisticas_equipo
* golstats.gold.estadisticas_jugador

###Matches validation

In [0]:
from pyspark.sql.functions import count, countDistinct

# Gold matches validation
df_partidos = spark.table("golstats.gold.partidos")

display(
    df_partidos.agg(
        count("*").alias("total_matches"),
        countDistinct("match_id").alias("unique_matches")
    )
)

###Team statistics validation

In [0]:
df_team = spark.table("golstats.gold.estadisticas_equipo")

display(
    df_team.agg(
        count("*").alias("total_team_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("team").alias("total_teams"),
        countDistinct(
            "match_id",
            "team"
        ).alias("unique_team_matches")
    )
)

###Player statistics validation

In [0]:
df_player = spark.table("golstats.gold.estadisticas_jugador")

display(
    df_player.agg(
        count("*").alias("total_player_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("player_id").alias("unique_players")
    )
)

In [0]:
display(
    df_player
    .filter(col("player") == "Lionel Andrés Messi Cuccittini")
    .orderBy("match_id")
)

In [0]:
display(
    df_player
    .groupBy("match_id", "player_id")
    .count()
    .filter("count > 1")
)

##16. Create Tournament-Level Analytics

With the complete Gold layer validated, the next phase focuses on tournament-level analysis.

The objective is to aggregate the match-level and player-level metrics into competition-wide insights.

The analysis will focus on:

* team performance;
* attacking performance;
* passing performance;
* defensive activity;
* player performance;
* expected goals.

The Gold tables will remain the source of truth for all downstream analytical queries.

In [0]:
from pyspark.sql.functions import (
    col,
    countDistinct,
    desc,
    round,
    sum,
    when
)

team_tournament = (
    spark.table("golstats.gold.estadisticas_equipo")
    .groupBy("team")
    .agg(
        countDistinct("match_id").alias("matches_played"),

        sum("goals").alias("goals"),

        round(sum("xg"), 2).alias("total_xg"),

        sum("shots").alias("total_shots"),

        sum("passes").alias("total_passes"),

        sum("completed_passes").alias("completed_passes"),

        sum("pressures").alias("total_pressures"),

        sum("ball_recoveries").alias("total_ball_recoveries"),

        sum("dispossessions").alias("total_dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        round(
            when(
                col("total_passes") > 0,
                col("completed_passes") /
                col("total_passes") * 100
            ).otherwise(0),
            2
        )
    )
)

display(
    team_tournament.orderBy(
        desc("goals")
    )
)

### 16.2 Build player tournament statistics

The player-level Gold dataset contains one record per player per match.

The next step is to aggregate these match-level statistics into tournament-level player performance metrics.

The aggregation will provide a competition-wide view of individual player performance.

The following metrics will be calculated:

* `matches_played`
* `goals`
* `total_xg`
* `total_shots`
* `total_passes`
* `completed_passes`
* `pass_completion_pct`
* `total_pressures`
* `total_ball_recoveries`
* `total_dispossessions`

The aggregation will use the Gold player statistics table as its source of truth.

The resulting dataset will have the following analytical grain:

**One row = one player in the tournament.**

Players will be identified using `player_id` and `player`.


In [0]:
from pyspark.sql.functions import (
    col,
    countDistinct,
    desc,
    round,
    sum,
    when
)

player_tournament = (
    spark.table("golstats.gold.estadisticas_jugador")
    .groupBy(
        "player_id",
        "player"
    )
    .agg(
        countDistinct("match_id").alias("matches_played"),

        sum("goals").alias("goals"),

        round(
            sum("xg"),
            2
        ).alias("total_xg"),

        sum("shots").alias("total_shots"),

        sum("passes").alias("total_passes"),

        sum("completed_passes").alias("completed_passes"),

        sum("pressures").alias("total_pressures"),

        sum("ball_recoveries").alias("total_ball_recoveries"),

        sum("dispossessions").alias("total_dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        round(
            when(
                col("total_passes") > 0,
                col("completed_passes") /
                col("total_passes") * 100
            ).otherwise(0),
            2
        )
    )
)

display(
    player_tournament
    .orderBy(desc("goals"))
)

### 16.2 Build player tournament summary

The player-match Gold dataset has now been validated and persisted for the
complete FIFA World Cup 2022 competition.

The next analytical dataset will aggregate player-match statistics across the
entire tournament.

The analytical grain will be:

**One row = one player across the tournament.**

The following metrics will be aggregated:

* `matches_played`
* `total_events`
* `passes`
* `completed_passes`
* `pass_completion_pct`
* `shots`
* `goals`
* `xg`
* `pressures`
* `ball_recoveries`
* `dispossessions`

The dataset will be used to analyze player performance at tournament level
and will later serve as a source for Power BI visualizations.

The aggregation will use the validated Gold table:

`golstats.gold.estadisticas_jugador`


In [0]:
from pyspark.sql.functions import (
    col,
    countDistinct,
    round,
    sum,
    when
)

player_tournament = (
    spark.table("golstats.gold.estadisticas_jugador")
    .groupBy(
        "player_id",
        "player"
    )
    .agg(
        countDistinct("match_id").alias("matches_played"),
        sum("total_events").alias("total_events"),
        sum("passes").alias("passes"),
        sum("completed_passes").alias("completed_passes"),
        sum("shots").alias("shots"),
        sum("goals").alias("goals"),
        round(sum("xg"), 2).alias("total_xg"),
        sum("pressures").alias("total_pressures"),
        sum("ball_recoveries").alias("total_ball_recoveries"),
        sum("dispossessions").alias("total_dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        round(
            when(
                col("passes") > 0,
                col("completed_passes") / col("passes") * 100
            ).otherwise(0),
            2
        )
    )
)

In [0]:
display(
    player_tournament
    .orderBy(col("goals").desc(), col("total_xg").desc())
)

### 16.2.1 Investigate player goal discrepancy

The tournament-level player aggregation revealed an unexpected result: some
players have more goals in the event data than their official tournament
statistics.

For example, Lionel Messi appears with 9 goals in the current aggregation,
while his official FIFA World Cup 2022 total was 7 goals.

This discrepancy may be caused by penalty shootout events being represented
as goal events in the StatsBomb data.

Before modifying the analytical logic, the underlying goal events will be
investigated directly.

The investigation will examine:

* The match associated with each goal event.
* The event period.
* The event timestamp.
* The shot outcome.
* The shot technique.
* The event index.

The objective is to determine whether penalty shootout events are being
included in the player goal metric.

If confirmed, the Gold event data will remain unchanged because the events
are valid StatsBomb records. Instead, the analytical definition of
`tournament goals` will be refined to count goals scored during regular
time and extra time, excluding penalty shootout goals.


In [0]:
display(
    df_silver
    .filter(
        (col("player") == "Lionel Andrés Messi Cuccittini") &
        (col("event_type") == "Shot") &
        (col("shot_outcome") == "Goal")
    )
    .select(
        "match_id",
        "period",
        "minute",
        "second",
        "index",
        "shot_technique",
        "shot_xg",
        "shot_outcome"
    )
    .orderBy("match_id", "period", "index")
)

### 16.2.2 Investigate penalty shootout goals

The previous investigation confirmed that StatsBomb represents penalty shootout
attempts as shot events in `period = 5`.

These events are valid event records and will remain in the Silver layer.

However, penalty shootout goals must not be included in the tournament goal
statistics because they are not goals scored during regular time or extra time.

Before modifying the Gold aggregation, the complete competition dataset will be
checked to determine how many shootout goals are currently classified as goals.

This provides a global validation of the issue and ensures that the analytical
rule is applied consistently across the competition.


In [0]:
display(
    df_silver
    .filter(
        (col("event_type") == "Shot") &
        (col("shot_outcome") == "Goal")
    )
    .groupBy("period")
    .count()
    .orderBy("period")
)

### 16.2.3 Reconcile tournament goals

The global analysis identified 26 goal events recorded in penalty shootout
periods.

These events must be excluded from tournament goal statistics because penalty
shootouts are separate from goals scored during the match.

However, the current event-based goal count also reveals a discrepancy between
the number of goals represented by `Shot` events and the expected tournament
total.

Goals from periods 1 through 4 should represent goals scored during regular
time and extra time, while period 5 represents penalty shootouts.

Before updating the Gold analytical logic, the remaining discrepancy will be
investigated at match level.

The objective is to identify whether some goals are represented through
different StatsBomb event types, such as own goals, rather than standard
`Shot` events.

This reconciliation will ensure that the final Gold goal metric is complete
and does not depend exclusively on standard shot events.


In [0]:
display(
    df_silver
    .filter(
        (col("event_type") == "Shot") &
        (col("shot_outcome") == "Goal") &
        (col("period") <= 4)
    )
    .groupBy("match_id")
    .count()
    .orderBy("match_id")
)

In [0]:
display(
    df_silver
    .filter(col("event_type") == "Own Goal For")
    .select(
        "match_id",
        "period",
        "minute",
        "second",
        "team",
        "player_id",
        "player"
    )
    .orderBy("match_id", "index")
)

### 16.2.4 Define tournament goal counting logic

The investigation reconciled the tournament goal events across the complete
competition.

StatsBomb contains 195 events classified as `Shot` with outcome `Goal`.

However, 26 of these events occur in `period = 5`, representing penalty
shootouts. These attempts must not be included in match goal statistics.

The remaining 169 goals occur during regular time or extra time.

In addition, the dataset contains 3 `Own Goal For` events. These represent
goals scored against a team as a result of an own goal and therefore must be
included in team goal statistics.

The resulting tournament goal reconciliation is:

* 169 goals from regular time and extra time shots.
* 3 own goals.
* 172 total match goals.
* 26 penalty shootout goals excluded from tournament goal totals.

The analytical definitions will therefore be:

**Player goals**

Count `Shot` events with outcome `Goal` during periods 1 through 4.

**Team goals**

Count `Shot` events with outcome `Goal` during periods 1 through 4,
plus goals received from `Own Goal For` events.

Penalty shootout goals are excluded from both player and team tournament
goal statistics.

The Silver layer remains unchanged because all underlying StatsBomb events
are valid and should be preserved.


In [0]:
df_player_match_corrected = (
    df_silver
    .filter(col("player_id").isNotNull())
    .groupBy(
        "match_id",
        "team",
        "player_id",
        "player"
    )
    .agg(
        count("*").alias("total_events"),

        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal") &
                (col("period").isin([1, 2, 3, 4])),
                True
            )
        ).alias("goals"),

        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        count(
            when(col("event_type") == "Pressure", True)
        ).alias("pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

In [0]:
display(
    df_player_match_corrected
    .filter(col("player") == "Lionel Andrés Messi Cuccittini")
    .select(
        "match_id",
        "goals",
        "shots",
        "xg"
    )
    .orderBy("match_id")
)

In [0]:
display(
    df_player_match_corrected
    .filter(col("player") == "Kylian Mbappé Lottin")
    .select(
        "match_id",
        "goals",
        "shots",
        "xg"
    )
    .orderBy("match_id")
)

In [0]:
display(
    df_player_match_corrected.agg(
        sum("goals").alias("total_player_goals")
    )
)

In [0]:
%sql
DROP TABLE IF EXISTS golstats.gold.estadisticas_jugador;

In [0]:
PLAYER_GOLD_TABLE = "golstats.gold.estadisticas_jugador"

(
    df_player_match_corrected
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PLAYER_GOLD_TABLE)
)

print(f"Gold player table persisted: {PLAYER_GOLD_TABLE}")

In [0]:
df_player_gold = spark.table("golstats.gold.estadisticas_jugador")

display(
    df_player_gold.agg(
        count("*").alias("total_player_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("player_id").alias("unique_players"),
        sum("goals").alias("total_goals")
    )
)

### 16.2.5 Validate player goal reconciliation

The corrected player-match Gold dataset has been validated after excluding
penalty shootout goals.

The final dataset contains:

* 1,996 player-match records.
* 64 matches.
* 680 unique players.
* 169 player-attributed goals.

The 169 goals correspond to goals scored by players during regular time and
extra time.

The three own goals identified in the competition are not included in the
player goal total because StatsBomb does not attribute these events to a
`player_id`.

The 26 penalty shootout goals are also excluded from player tournament goal
statistics.

The player-match Gold dataset therefore preserves the intended analytical
definition while keeping all underlying event records available in Silver.


### 16.3 Define team goal attribution

The team-match Gold dataset requires a different goal definition from the
player-match dataset.

Player statistics count only goals directly attributed to a player through
`Shot` events.

Team statistics must also include own goals because an own goal contributes
to the opponent's score.

The final team goal definition is therefore:

**Team goals =**

* `Shot` events with outcome `Goal` during periods 1 through 4.
* Plus goals resulting from `Own Goal For` events, attributed to the opponent
  of the team recorded in the event.

Penalty shootout goals from period 5 are excluded.

The three identified own goals will therefore be attributed to the opposing
team using the official match metadata rather than hardcoded team names.

This preserves a reusable analytical rule that can be applied to the complete
competition dataset.


In [0]:
display(
    matches_complete_df
    .filter(
        col("match_id").isin(
            3857276,
            3857292,
            3869151
        )
    )
    .orderBy("match_id")
)

### 16.3.1 Calculate correct team goals

The team-match Gold dataset will now be rebuilt using the validated tournament
goal definition.

Regular and extra-time goals are identified through `Shot` events with
`shot_outcome = Goal` and periods 1 through 4.

Own goals require separate treatment. In StatsBomb, an `Own Goal For` event is
associated with the team that conceded the own goal rather than the team that
receives the goal.

Therefore, each own goal must be attributed to the opponent of the team
recorded in the event.

The opponent will be determined dynamically from the official match metadata:

* If the event team is the home team, the goal is assigned to the away team.
* If the event team is the away team, the goal is assigned to the home team.

This avoids hardcoded team mappings and makes the transformation reusable.

Penalty shootout goals from period 5 remain excluded.


In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    when,
    coalesce,
    lit
)

df_team_goals = (
    df_silver
    .filter(
        (col("event_type") == "Shot") &
        (col("shot_outcome") == "Goal") &
        (col("period").isin([1, 2, 3, 4]))
    )
    .groupBy("match_id", "team")
    .agg(
        count("*").alias("match_goals")
    )
)

display(
    df_team_goals
    .orderBy("match_id", "team")
)

In [0]:
df_own_goals = (
    df_silver
    .filter(col("event_type") == "Own Goal For")
    .select(
        "match_id",
        "team"
    )
    .join(
        matches_complete_df.select(
            "match_id",
            "home_team",
            "away_team"
        ),
        on="match_id",
        how="left"
    )
)

In [0]:
df_own_goals = (
    df_own_goals
    .withColumn(
        "goal_team",
        when(
            col("team") == col("home_team"),
            col("away_team")
        ).otherwise(col("home_team"))
    )
)

display(df_own_goals)

In [0]:
df_own_goal_counts = (
    df_own_goals
    .groupBy(
        "match_id",
        col("goal_team").alias("team")
    )
    .agg(
        count("*").alias("own_goals_received")
    )
)

display(df_own_goal_counts)

In [0]:
df_team_match_corrected = (
    df_team_match
    .join(
        df_own_goal_counts,
        on=["match_id", "team"],
        how="left"
    )
    .withColumn(
        "goals",
        col("goals") + coalesce(col("own_goals_received"), lit(0))
    )
    .drop("own_goals_received")
)

In [0]:
display(
    df_team_match_corrected.agg(
        sum("goals").alias("total_team_goals")
    )
)

### 16.3.2 Rebuild team goals using the validated definition

The previous team-match dataset contains 195 goals because its original
aggregation counted all `Shot` events with outcome `Goal`, including the
26 penalty shootout goals.

Adding own goals to that existing metric would therefore produce an incorrect
total of 198 goals.

Instead, the team goal metric will be rebuilt from the underlying Silver
events using the validated analytical definition.

The corrected calculation will:

1. Count `Shot` events with outcome `Goal` during periods 1 through 4.
2. Exclude all period 5 penalty shootout goals.
3. Add `Own Goal For` events to the opponent's team.
4. Preserve all team-match records, including teams with zero goals.

The resulting team goal total is expected to be 172.


In [0]:
display(
    df_team_goals.agg(
        sum("match_goals").alias("regular_extra_time_goals")
    )
)

In [0]:
df_team_match_base = df_team_match.drop("goals")

In [0]:
df_team_match_corrected = (
    df_team_match_base
    .join(
        df_team_goals,
        on=["match_id", "team"],
        how="left"
    )
    .withColumn(
        "goals",
        coalesce(col("match_goals"), lit(0))
    )
    .drop("match_goals")
)

In [0]:
df_team_match_corrected = (
    df_team_match_corrected
    .join(
        df_own_goal_counts,
        on=["match_id", "team"],
        how="left"
    )
    .withColumn(
        "goals",
        col("goals") + coalesce(col("own_goals_received"), lit(0))
    )
    .drop("own_goals_received")
)

In [0]:
display(
    df_team_match_corrected.agg(
        sum("goals").alias("total_team_goals")
    )
)

### 16.3.3 Reconcile team goals against official match results

The corrected team-match Gold dataset now contains 172 tournament goals.

The next validation compares the team-level goal totals against the official
match metadata for every match in the competition.

For each match, the validation will compare:

* Official home team score.
* Calculated home team goals.
* Official away team score.
* Calculated away team goals.

The validation must return zero discrepancies.

This provides a match-level reconciliation rather than relying only on the
global tournament goal total.


In [0]:
df_team_goals_validation = (
    df_team_match_corrected
    .select(
        "match_id",
        "team",
        "goals"
    )
)

In [0]:
df_match_reconciliation = (
    matches_complete_df
    .join(
        df_team_goals_validation,
        (matches_complete_df.match_id == df_team_goals_validation.match_id) &
        (matches_complete_df.home_team == df_team_goals_validation.team),
        "left"
    )
    .withColumnRenamed("goals", "calculated_home_goals")
    .drop(df_team_goals_validation.match_id)
    .drop(df_team_goals_validation.team)
)

In [0]:
df_home_goals = (
    df_team_match_corrected
    .select(
        "match_id",
        col("team").alias("home_team"),
        col("goals").alias("calculated_home_goals")
    )
)

df_away_goals = (
    df_team_match_corrected
    .select(
        "match_id",
        col("team").alias("away_team"),
        col("goals").alias("calculated_away_goals")
    )
)

In [0]:
df_match_reconciliation = (
    matches_complete_df
    .join(
        df_home_goals,
        on=["match_id", "home_team"],
        how="left"
    )
    .join(
        df_away_goals,
        on=["match_id", "away_team"],
        how="left"
    )
)

In [0]:
df_match_discrepancies = (
    df_match_reconciliation
    .filter(
        (col("home_score") != col("calculated_home_goals")) |
        (col("away_score") != col("calculated_away_goals"))
    )
)

display(
    df_match_discrepancies.select(
        "match_id",
        "home_team",
        "home_score",
        "calculated_home_goals",
        "away_team",
        "away_score",
        "calculated_away_goals"
    )
)

### 16.3.4 Correct interpretation of `Own Goal For`

The previous reconciliation identified a discrepancy in matches containing
`Own Goal For` events.

The investigation showed that the `team` associated with a StatsBomb
`Own Goal For` event already represents the team receiving the goal.

Therefore, the previous logic that assigned the goal to the opposing team was
incorrect.

The corrected rule is:

* `Shot + Goal` in periods 1–4 → goal assigned to the event team.
* `Own Goal For` → goal assigned directly to the event team.
* Period 5 shootout goals → excluded from match goals.

The Silver layer remains unchanged because the underlying event data is valid.

Only the Team Gold goal calculation needs to be corrected.


In [0]:
df_own_goal_counts = (
    df_silver
    .filter(col("event_type") == "Own Goal For")
    .groupBy(
        "match_id",
        "team"
    )
    .agg(
        count("*").alias("own_goals_received")
    )
)

display(df_own_goal_counts.orderBy("match_id"))

In [0]:
df_team_match_base = df_team_match.drop("goals")

df_team_match_corrected = (
    df_team_match_base
    .join(
        df_team_goals,
        on=["match_id", "team"],
        how="left"
    )
    .withColumn(
        "goals",
        coalesce(col("match_goals"), lit(0))
    )
    .drop("match_goals")
)

df_team_match_corrected = (
    df_team_match_corrected
    .join(
        df_own_goal_counts,
        on=["match_id", "team"],
        how="left"
    )
    .withColumn(
        "goals",
        col("goals") + coalesce(col("own_goals_received"), lit(0))
    )
    .drop("own_goals_received")
)

In [0]:
display(
    df_team_match_corrected.agg(
        sum("goals").alias("total_team_goals")
    )
)

In [0]:
df_home_goals = (
    df_team_match_corrected
    .select(
        "match_id",
        col("team").alias("home_team"),
        col("goals").alias("calculated_home_goals")
    )
)

df_away_goals = (
    df_team_match_corrected
    .select(
        "match_id",
        col("team").alias("away_team"),
        col("goals").alias("calculated_away_goals")
    )
)

df_match_reconciliation = (
    matches_complete_df
    .join(
        df_home_goals,
        on=["match_id", "home_team"],
        how="left"
    )
    .join(
        df_away_goals,
        on=["match_id", "away_team"],
        how="left"
    )
)

df_match_discrepancies = (
    df_match_reconciliation
    .filter(
        (col("home_score") != col("calculated_home_goals")) |
        (col("away_score") != col("calculated_away_goals"))
    )
)

display(
    df_match_discrepancies.select(
        "match_id",
        "home_team",
        "home_score",
        "calculated_home_goals",
        "away_team",
        "away_score",
        "calculated_away_goals"
    )
)

### 16.3.5 Final team-goal reconciliation

The corrected Team Gold dataset has been reconciled against the official match
metadata for all 64 FIFA World Cup 2022 matches.

The validation compared the calculated home and away goals against the official
match scores for every match.

The reconciliation returned zero discrepancies.

Therefore, the goal calculation is considered validated:

* Regular goals: `Shot + Goal` during periods 1–4.
* Own goals: `Own Goal For`, assigned to the event team.
* Penalty shootout goals: period 5, excluded from match goals.

The resulting tournament total is:

**172 match goals**

The Team Gold dataset is now ready to be persisted.


In [0]:
%sql
DROP TABLE IF EXISTS golstats.gold.estadisticas_equipo;

In [0]:
TEAM_GOLD_TABLE = "golstats.gold.estadisticas_equipo"

(
    df_team_match_corrected
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TEAM_GOLD_TABLE)
)

In [0]:
df_team = spark.table("golstats.gold.estadisticas_equipo")

display(
    df_team.agg(
        count("*").alias("total_team_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("team").alias("total_teams"),
        sum("goals").alias("total_goals")
    )
)

## 17. Validate Player Gold

The Player Gold dataset has already been rebuilt using the complete 64-match
Silver dataset.

Player statistics are calculated only from events with a valid `player_id`.

Goals are defined as:

* `Shot + Goal` during periods 1–4.
* Period 5 shootout goals are excluded.
* `Own Goal For` events are excluded because they do not have individual player
  attribution.

Therefore, Player Gold is expected to contain 169 goals, while Team Gold
contains 172 goals.

The three-goal difference corresponds to the three own goals recorded in the
competition.

The persisted Player Gold table will now be validated to ensure that the
corrected dataset was successfully stored.


In [0]:
df_player = spark.table("golstats.gold.estadisticas_jugador")

display(
    df_player.agg(
        count("*").alias("total_player_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("player_id").alias("unique_players"),
        sum("goals").alias("total_player_goals")
    )
)

In [0]:
display(
    df_player
    .filter(
        col("player").isin([
            "Lionel Andrés Messi Cuccittini",
            "Kylian Mbappé Lottin"
        ])
    )
    .groupBy("player")
    .agg(
        countDistinct("match_id").alias("matches_played"),
        sum("goals").alias("goals"),
        sum("shots").alias("shots"),
        sum("xg").alias("xg"),
        sum("passes").alias("passes")
    )
)

## 18. Build Player Tournament Analytics

The Player Match Gold dataset contains one row per player per match.

For tournament-level analysis, these records will be aggregated into one row per
player across the complete FIFA World Cup 2022 competition.

The resulting dataset will provide cumulative player performance metrics that
can be consumed directly by analytical tools such as Power BI.

The aggregation will include:

* Matches played.
* Total events.
* Passes.
* Completed passes.
* Pass completion percentage.
* Shots.
* Goals.
* xG.
* Pressures.
* Ball recoveries.
* Dispossessions.

The analytical grain will be:

**One row = one player in the tournament.**

The tournament-level pass completion percentage will be calculated from the
aggregated number of completed passes and total passes rather than averaging
the percentages from individual matches.

This avoids weighting every match equally when the number of passes differs
between matches.


In [0]:
from pyspark.sql.functions import col, countDistinct, sum, round, when

df_player_tournament = (
    df_player
    .groupBy(
        "player_id",
        "player"
    )
    .agg(
        countDistinct("match_id").alias("matches_played"),
        sum("total_events").alias("total_events"),
        sum("passes").alias("passes"),
        sum("completed_passes").alias("completed_passes"),
        sum("shots").alias("shots"),
        sum("goals").alias("goals"),
        sum("xg").alias("xg"),
        sum("pressures").alias("pressures"),
        sum("ball_recoveries").alias("ball_recoveries"),
        sum("dispossessions").alias("dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

In [0]:
display(
    df_player_tournament
    .orderBy(col("goals").desc(), col("xg").desc())
)

In [0]:
display(
    df_player_tournament.agg(
        count("*").alias("total_players"),
        countDistinct("player_id").alias("unique_players"),
        sum("goals").alias("total_goals"),
        sum("shots").alias("total_shots"),
        sum("passes").alias("total_passes"),
        sum("pressures").alias("total_pressures"),
        sum("ball_recoveries").alias("total_ball_recoveries"),
        sum("dispossessions").alias("total_dispossessions")
    )
)

In [0]:
display(
    df_player_tournament
    .groupBy("player_id")
    .count()
    .filter(col("count") > 1)
)

### 18.3 Investigate duplicated player records

The Player Tournament dataset is expected to have one row per player across
the entire competition.

The validation identified one duplicated player identifier:

* `player_id = 4354`

Before applying any correction, the underlying records will be inspected to
determine why the same player appears more than once.

This investigation will compare the duplicated records across:

* Player name
* Team
* Matches played
* Goals
* Passes
* Shots
* xG
* Other aggregated metrics

The objective is to determine whether the duplication represents a data
quality issue or a legitimate difference in player identity attribution.


In [0]:
display(
    df_player_tournament
    .filter(col("player_id") == 4354)
)

In [0]:
display(
    df_player
    .filter(col("player_id") == 4354)
    .orderBy("match_id")
)

### 18.3.1 Normalize player identity

The investigation identified that player `4354` appears with two name variants:

* `Phil Foden`
* `Philip Foden`

Both records refer to the same player and share the same StatsBomb
`player_id`.

Therefore, the player identifier is considered the canonical key for the
Player Tournament dataset.

The previous aggregation grouped by both `player_id` and `player`, which caused
the same player to appear as two separate tournament records.

The Player Tournament aggregation will therefore be rebuilt using only
`player_id` as the grouping key.

A representative player name will then be associated with each player ID.

This ensures the analytical grain:

**One row = one unique player in the tournament.**


In [0]:
from pyspark.sql.functions import first

df_player_tournament = (
    df_player
    .groupBy("player_id")
    .agg(
        first("player").alias("player"),
        countDistinct("match_id").alias("matches_played"),
        sum("total_events").alias("total_events"),
        sum("passes").alias("passes"),
        sum("completed_passes").alias("completed_passes"),
        sum("shots").alias("shots"),
        sum("goals").alias("goals"),
        sum("xg").alias("xg"),
        sum("pressures").alias("pressures"),
        sum("ball_recoveries").alias("ball_recoveries"),
        sum("dispossessions").alias("dispossessions")
    )
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

In [0]:
display(
    df_player_tournament
    .filter(col("player_id") == 4354)
)

In [0]:
display(
    df_player_tournament.agg(
        count("*").alias("total_players"),
        countDistinct("player_id").alias("unique_players"),
        sum("goals").alias("total_goals"),
        sum("shots").alias("total_shots"),
        sum("passes").alias("total_passes"),
        sum("pressures").alias("total_pressures"),
        sum("ball_recoveries").alias("total_ball_recoveries"),
        sum("dispossessions").alias("total_dispossessions")
    )
)

### 18.4 Validate Player Tournament grain

The Player Tournament dataset has been reconciled against the Player Match Gold
dataset.

The aggregated metrics remain consistent, and the number of player records now
matches the number of unique player identifiers.

The final grain validation will confirm that each `player_id` appears exactly
once in the tournament dataset.

Expected result:

**No duplicated player identifiers.**


In [0]:
display(
    df_player_tournament
    .groupBy("player_id")
    .count()
    .filter(col("count") > 1)
)

### 18.5 Persist Player Tournament

The Player Tournament dataset has passed all structural and metric validations.

The final dataset contains:

* 680 unique players.
* One row per player.
* 169 player-attributed goals.
* Tournament-level aggregated performance metrics.

The validated dataset will now be persisted as a Delta table for downstream
analytical consumption.

The table will be stored in the Gold layer as:

`golstats.gold.player_tournament`


In [0]:
%sql
DROP TABLE IF EXISTS golstats.gold.player_tournament;

In [0]:
PLAYER_TOURNAMENT_TABLE = "golstats.gold.player_tournament"

(
    df_player_tournament
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PLAYER_TOURNAMENT_TABLE)
)

In [0]:
df_player_tournament_final = spark.table(
    "golstats.gold.player_tournament"
)

display(
    df_player_tournament_final.agg(
        count("*").alias("total_players"),
        countDistinct("player_id").alias("unique_players"),
        sum("goals").alias("total_goals"),
        sum("shots").alias("total_shots"),
        sum("passes").alias("total_passes")
    )
)

## 19. Rebuild Match Gold

The Match Gold dataset currently contains the 10-match development dataset.

Since the Bronze and Silver layers, as well as the Team and Player Gold datasets,
have now been scaled to the complete FIFA World Cup 2022 competition, the Match
Gold dataset must also be rebuilt.

The complete match metadata previously validated in this notebook will be used
as the source.

The resulting dataset will contain:

* One row per match.
* Official match date.
* Home team.
* Away team.
* Home score.
* Away score.
* Match result.

The analytical grain will remain:

**One row = one match.**

The expected result is a complete dataset containing all 64 FIFA World Cup 2022
matches.


In [0]:
from pyspark.sql.functions import col, when

df_matches_gold = (
    matches_complete_df
    .select(
        "match_id",
        "match_date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    )
    .withColumn(
        "result",
        when(col("home_score") > col("away_score"), "Home")
        .when(col("home_score") < col("away_score"), "Away")
        .otherwise("Draw")
    )
)

In [0]:
display(
    df_matches_gold.agg(
        count("*").alias("total_matches"),
        countDistinct("match_id").alias("unique_match_ids"),
        sum("home_score").alias("total_home_goals"),
        sum("away_score").alias("total_away_goals")
    )
)

In [0]:
display(
    df_matches_gold
    .groupBy("result")
    .count()
    .orderBy("result")
)

### 19.2 Reconcile Match Gold with Team Gold

The Match Gold dataset contains the official match scores obtained from the
StatsBomb competition metadata.

The Team Gold dataset independently calculates team goals from event-level data.

These two datasets should produce identical scores for every match.

The reconciliation will compare:

* Official home score vs calculated home goals.
* Official away score vs calculated away goals.

The goal calculation in Team Gold excludes penalty shootout goals and correctly
handles `Own Goal For` events.

The expected result is:

**No discrepancies between Match Gold and Team Gold.**

This validation provides an important cross-layer consistency check between
official match metadata and event-derived analytical metrics.


In [0]:
id="v8c2nm"
df_home_goals = (
    df_team_match_corrected
    .select(
        "match_id",
        col("team").alias("home_team"),
        col("goals").alias("calculated_home_goals")
    )
)

df_away_goals = (
    df_team_match_corrected
    .select(
        "match_id",
        col("team").alias("away_team"),
        col("goals").alias("calculated_away_goals")
    )
)

df_match_reconciliation = (
    df_matches_gold
    .join(
        df_home_goals,
        on=["match_id", "home_team"],
        how="left"
    )
    .join(
        df_away_goals,
        on=["match_id", "away_team"],
        how="left"
    )
)

In [0]:
id="q3w7ka"
df_match_discrepancies = (
    df_match_reconciliation
    .filter(
        (col("home_score") != col("calculated_home_goals")) |
        (col("away_score") != col("calculated_away_goals"))
    )
)

display(
    df_match_discrepancies.select(
        "match_id",
        "home_team",
        "home_score",
        "calculated_home_goals",
        "away_team",
        "away_score",
        "calculated_away_goals"
    )
)

### 19.3 Persist Match Gold

The Match Gold dataset has been successfully reconciled against the Team Gold
dataset for all 64 matches.

No score discrepancies were found between the official match metadata and the
event-derived team statistics.

The validated dataset will now replace the previous 10-match development
version and will be persisted as a Delta table.

The final table will contain:

* 64 matches.
* 64 unique match identifiers.
* Official home and away scores.
* Match result.
* One row per match.

The table will be stored as:

`golstats.gold.partidos`


In [0]:
%sql
DROP TABLE IF EXISTS golstats.gold.partidos;

In [0]:
MATCH_GOLD_TABLE = "golstats.gold.partidos"

(
    df_matches_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(MATCH_GOLD_TABLE)
)

In [0]:
df_matches_final = spark.table(
    "golstats.gold.partidos"
)

display(
    df_matches_final.agg(
        count("*").alias("total_matches"),
        countDistinct("match_id").alias("unique_match_ids"),
        sum("home_score").alias("total_home_goals"),
        sum("away_score").alias("total_away_goals")
    )
)

## 20. Final Gold Quality Validation

All Gold datasets have now been rebuilt using the complete FIFA World Cup 2022
competition.

The final validation consolidates the main structural and business rules
verified throughout the scaling process.

The following controls will be performed:

* Match Gold contains exactly 64 matches.
* Team Gold contains exactly two teams per match.
* Player Gold contains no null player identifiers.
* Player Gold contains no duplicate player-match combinations.
* Player Tournament contains exactly one row per player.
* Player Tournament contains no duplicate player identifiers.
* Team Gold goals reconcile with Match Gold scores.
* Player-attributed goals reconcile with the event-level goal definition.
* No invalid pass completion percentages are present.

The objective is to establish a final quality gate before the dataset is exposed
to downstream analytical tools such as Power BI and Machine Learning.

If all validations pass, the Gold layer will be considered production-ready for
the portfolio project.


In [0]:
from pyspark.sql.functions import col, count, countDistinct, sum

df_partidos = spark.table("golstats.gold.partidos")
df_team = spark.table("golstats.gold.estadisticas_equipo")
df_player = spark.table("golstats.gold.estadisticas_jugador")
df_player_tournament = spark.table("golstats.gold.player_tournament")

checks = []

# 1. Match Gold = 64 partidos
n = df_partidos.select(countDistinct("match_id")).first()[0]
checks.append(("Match Gold contiene 64 partidos", n == 64, n))

# 2. Team Gold = 2 equipos por partido
bad = (
    df_team
    .groupBy("match_id")
    .agg(countDistinct("team").alias("n"))
    .filter(col("n") != 2)
    .count()
)
checks.append(("Team Gold: 2 equipos por partido", bad == 0, bad))

# 3. Player Gold sin player_id nulo
n = df_player.filter(col("player_id").isNull()).count()
checks.append(("Player Gold sin player_id nulo", n == 0, n))

# 4. Player Gold sin duplicados jugador-partido
n = (
    df_player
    .groupBy("match_id", "player_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
checks.append(("Player Gold sin duplicados jugador-partido", n == 0, n))

# 5. Player Tournament: una fila por jugador
n = (
    df_player_tournament
    .groupBy("player_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
checks.append(("Player Tournament sin duplicados", n == 0, n))

# 6. Team Gold vs Match Gold: goles reconciliados
recon = (
    df_partidos.alias("m")
    .join(
        df_team.alias("h"),
        (col("m.match_id") == col("h.match_id")) &
        (col("m.home_team") == col("h.team"))
    )
    .join(
        df_team.alias("a"),
        (col("m.match_id") == col("a.match_id")) &
        (col("m.away_team") == col("a.team"))
    )
    .filter(
        (col("m.home_score") != col("h.goals")) |
        (col("m.away_score") != col("a.goals"))
    )
    .count()
)

checks.append(("Team Gold reconciliado con Match Gold", recon == 0, recon))

# 7. Pass completion % dentro de rango válido
n = (
    df_team
    .filter(
        (col("pass_completion_pct") < 0) |
        (col("pass_completion_pct") > 100)
    )
    .count()
)

checks.append(("Pass completion % válido (0-100)", n == 0, n))

# 8. Player Gold: goles atribuidos correctamente
player_goals = (
    df_player
    .agg(sum("goals").alias("total_goals"))
    .first()["total_goals"]
)

checks.append((
    "Player Gold contiene 169 goles atribuidos",
    player_goals == 169,
    player_goals
))

# Mostrar resultados
for name, passed, value in checks:
    print(("✅" if passed else "❌"), name, "->", value)

# Resultado final
all_passed = all(passed for _, passed, _ in checks)

print("\nGOLD LAYER PRODUCTION-READY:", all_passed)

## Power BI Readiness

## Pipeline Status: Production Ready 

The complete Medallion pipeline has been successfully executed and validated using all 64 matches from the FIFA World Cup 2022.

### Final Quality Validation

The following validations passed successfully:

- ✅ Match Gold contains all 64 tournament matches.
- ✅ Team Gold contains exactly 2 teams per match.
- ✅ Player Gold contains no null player IDs.
- ✅ Player Gold contains no duplicate player-match records.
- ✅ Player Tournament contains one record per player.
- ✅ Team goals are fully reconciled with official match results.
- ✅ Pass completion percentage is within the valid range (0–100).
- ✅ Player-attributed goals total 169.

### Final Result

```text
GOLD LAYER PRODUCTION-READY: True